# Transfer learning: PaSST e HTS-AT nos quatro cenários

Consulta dos resultados consolidados com encoders congelados. Fine-tuning não foi realizado. Dados, pesos e gráficos completos ficam locais; este notebook e os relatórios científicos são versionados.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Execute a partir da raiz do projeto ou notebooks/'
sys.path.insert(0, str(ROOT))
RUN_ADDITIONAL_TRANSFER = False
PREPARE_DATA = False
SCENARIOS_TO_RUN = ['mono_cont', 'poly_disc', 'poly_cont']

## Execução explícita opcional

Deixe o controle False para consultar resultados sem treinar novamente. A execução completa baixa/verifica WAVs se PREPARE_DATA=True, extrai embeddings e treina 34 cabeças por cenário. Cenários concluídos são reutilizados e caches existentes devem passar pela verificação. O CLI foi usado para a execução registrada; a cópia executada deste notebook documenta a consulta.

In [ ]:
if RUN_ADDITIONAL_TRANSFER:
    from scripts.run_additional_transfer_learning import run
    run(SCENARIOS_TO_RUN, allow_training=True, prepare_data=PREPARE_DATA)
else:
    print('Execução desabilitada. A consulta abaixo não inicia treinamento.')

In [ ]:
reports = {}
for scenario in ['mono_disc', 'mono_cont', 'poly_disc', 'poly_cont']:
    stem = 'transfer_learning_consolidated' if scenario == 'mono_disc' else f'transfer_learning_{scenario}_consolidated'
    path = ROOT / 'results' / f'{stem}_summary.json'
    if path.exists():
        reports[scenario] = json.loads(path.read_text(encoding='utf-8'))
    else:
        print('Relatório ainda não disponível:', scenario)
rows = []
for scenario, report in reports.items():
    for model, info in report['models'].items():
        row = {'cenário': scenario, 'modelo': model, 'teste': report['split_counts']['test']}
        for metric in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall']:
            value = info['aggregate'][metric]
            row[metric + ' (%)'] = f"{100*value['mean']:.2f} ± {100*value['std']:.2f}"
        rows.append(row)
display(pd.DataFrame(rows))

In [ ]:
from IPython.display import display, Markdown, Image
for scenario, report in reports.items():
    display(Markdown(f'### {scenario}'))
    for model, info in report['models'].items():
        display(Markdown(f"**{model}**: {info['configuration']}; seed representativa pela validação: {info['representative_seed_by_validation']}"))
        display(pd.DataFrame(info['final_runs'])[['training_seed', 'epochs_run', 'best_epoch', 'elapsed_seconds', 'stop_reason']])
        figure = ROOT / report['session'] / model / 'mean_test_confusion.png'
        if figure.exists():
            display(Image(filename=str(figure)))

Média e desvio entre cinco treinamentos no mesmo teste; não representam variabilidade entre novas fontes. Cada cenário tem cabeça própria. Para inferência, carregar head e scaler juntos usando `src.models.consolidated_probe.load_consolidated_probe`. Mono Discrete teve rodada preliminar com consulta ao teste. O baseline histórico usa outro split e frontend, portanto a comparação é descritiva.